In [ ]:
import os
import urllib.request
import feedparser
import time

# Install feedparser if missing
try:
    import feedparser
except ImportError:
    !pip install -q feedparser
    import feedparser

print("🚀 Starting Data Harvester v2 (Robust Mode)...")

# 1. Configuration
# We use a broad search to ensure we get results
SEARCH_QUERY = "all:electron+AND+all:crystal"
MAX_PAPERS = 50
DOWNLOAD_FOLDER = "industry_data"

os.makedirs(DOWNLOAD_FOLDER, exist_ok=True)

# 2. The Search Function
base_url = 'http://export.arxiv.org/api/query?'
print(f"🕵️ Searching ArXiv for {MAX_PAPERS} scientific papers...")

url = f'{base_url}search_query={SEARCH_QUERY}&start=0&max_results={MAX_PAPERS}'
data = urllib.request.urlopen(url).read()
feed = feedparser.parse(data)

print(f"✅ Found {len(feed.entries)} papers. Beginning download...")

# 3. Download Loop (Fixed for stability)
count = 0
for entry in feed.entries:
    try:
        # --- THE FIX: Find link by TYPE, not Title ---
        pdf_link = None
        for link in entry.links:
            if link.get('type') == 'application/pdf':
                pdf_link = link.get('href')
                break

        # Fallback: If no "pdf" type, look for "pdf" in the URL
        if not pdf_link:
            for link in entry.links:
                if 'pdf' in link.get('href', ''):
                    pdf_link = link.get('href')
                    break

        if pdf_link:
            # ArXiv trick: change /abs/ to /pdf/ if needed
            if '/abs/' in pdf_link:
                pdf_link = pdf_link.replace('/abs/', '/pdf/')
            if not pdf_link.endswith('.pdf'):
                pdf_link += '.pdf'

            # Create a safe filename (remove special characters)
            safe_title = "".join([c if c.isalnum() else "_" for c in entry.title])[:50]
            filename = f"{safe_title}.pdf"
            filepath = os.path.join(DOWNLOAD_FOLDER, filename)

            # Download
            if not os.path.exists(filepath):
                urllib.request.urlretrieve(pdf_link, filepath)
                count += 1
                print(f"  [{count}/{MAX_PAPERS}] Downloaded: {entry.title[:30]}...")
                time.sleep(0.5) # Be polite
            else:
                print(f"  ⚠️ Skipping {entry.title[:20]}: No PDF link found.")

    except Exception as e:
        print(f"  ❌ Error on paper: {e}")

print("\n📦 HARVEST COMPLETE.")
print(f"You now have {count} Research Papers in the '{DOWNLOAD_FOLDER}' folder.")

🚀 Starting Data Harvester v2 (Robust Mode)...
🕵️ Searching ArXiv for 50 scientific papers...
✅ Found 50 papers. Beginning download...
  [1/50] Downloaded: Single Crystal Growth of Skutt...
  [2/50] Downloaded: Electron Beam Production by Py...
  [3/50] Downloaded: Pairing of valence electrons a...
  [4/50] Downloaded: Electron rest mass and energy ...
  ❌ Error on paper: HTTP Error 404: Not Found
  [5/50] Downloaded: Static and Dynamic Properties ...
  [6/50] Downloaded: Growth of Pure and Intercalate...
  [7/50] Downloaded: Quenching factor measurement f...
  [8/50] Downloaded: Tuning electric charge scatter...
  [9/50] Downloaded: Electronic Liquid Crystal Phas...
  [10/50] Downloaded: Synthesis and characterisation...
  [11/50] Downloaded: Multi-reference extension to v...
  [12/50] Downloaded: Low-frequency electromagnetic ...
  [13/50] Downloaded: Novel polymorphic phase of BaC...
  [14/50] Downloaded: The general traits of inelasti...
  [15/50] Downloaded: Spectral density functi

In [ ]:
# 1. Install the missing library
!pip install -q PyPDF2

import glob
import os
from PyPDF2 import PdfReader

print("⚙️ Processing the new Industry Data...")

# 2. Get the files
pdf_files = glob.glob("industry_data/*.pdf")

if not pdf_files:
    print("⚠️ WARNING: No PDFs found in 'industry_data'. Did the harvester run?")
else:
    print(f"✅ Found {len(pdf_files)} papers. Reading them now...")

    industry_data = []

    # 3. Read & Chunk
    for file_path in pdf_files:
        try:
            reader = PdfReader(file_path)
            text = ""
            # Extract text from every page
            for page in reader.pages:
                content = page.extract_text()
                if content:
                    text += content + "\n"

            # Chunking (2000 chars)
            chunk_size = 2000
            for i in range(0, len(text), chunk_size):
                chunk = text[i:i + chunk_size]
                # Only keep chunks with actual content
                if len(chunk) > 100:
                    industry_data.append({"text": f"### Knowledge: {chunk}"})

            print(f"  -> Processed: {os.path.basename(file_path)[:30]}...")

        except Exception as e:
            print(f"  ❌ Error reading {file_path}: {e}")

    print(f"\n✅ READY: Created {len(industry_data)} chunks from {len(pdf_files)} papers.")
    print("👉 You can now run the Trainer code!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 18.8 MB/s eta 0:00:00
⚙️ Processing the new Industry Data...
✅ Found 48 papers. Reading them now...
  -> Processed: Multi_reference_extension_to_v...
  -> Processed: Low_frequency_electromagnetic_...
  -> Processed: The_general_traits_of_inelasti...
  -> Processed: Impact_of_an_electron_Wigner_c...
  -> Processed: Metastable_electron_pair_state...
  -> Processed: MoS2_Dual_Gate_MOSFET_with_Ato...
  -> Processed: Electronic_Liquid_Crystal_Phas...
  -> Processed: Relativistic_electron_Wigner_c...
  -> Processed: Quantum_crystal_phase_of_elect...
  -> Processed: Normal_state_of_metal_intercal...
  -> Processed: Electronic_Phonons_in_a_Moiré_...
  -> Processed: Synthesis_and_characterisation...
  -> Processed: Pairing_of_valence_electrons_a...
  -> Processed: Electron_rest_mass_and_energy_...
  -> Processed: Sharp_Tunneling_Resonance_from...
  -> Processed: An_exciton_interacting_with_th...
  -> Processed: Tuning_electric_charge_sc

In [ ]:
import torch
import glob
from PyPDF2 import PdfReader
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

print("🚀 Starting Enterprise V2 Training...")

# --- 1. RELOAD DATA ---
print("📂 Loading data from 'industry_data' folder...")
pdf_files = glob.glob("industry_data/*.pdf")
industry_data = []

if len(pdf_files) == 0:
    print("⚠️ STOP: No PDFs found! Did you run the Harvester?")
else:
    for file_path in pdf_files:
        try:
            reader = PdfReader(file_path)
            text = ""
            for page in reader.pages:
                text += page.extract_text() or ""

            # Chunking
            chunk_size = 2000
            for i in range(0, len(text), chunk_size):
                chunk = text[i:i + chunk_size]
                if len(chunk) > 100:
                    industry_data.append({"text": f"### Knowledge: {chunk}"})
        except:
            pass
    print(f"✅ Data Loaded: {len(industry_data)} chunks ready for deep learning.")

    # --- 2. LOAD MODEL (Fresh Llama 3) ---
    print("🧠 Loading Base Llama 3 Model...")
    model_id = "unsloth/llama-3-8b-bnb-4bit"
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=bnb_config, device_map="auto", trust_remote_code=True
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token

    # --- 3. CONFIGURE ADAPTER (The "New Brain") ---
    model.gradient_checkpointing_enable()
    model = prepare_model_for_kbit_training(model)
    config = LoraConfig(
        r=16, lora_alpha=32, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, config)

    # --- 4. START TRAINING ---
    real_dataset = Dataset.from_list(industry_data)
    def tokenize_function(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

    tokenized_dataset = real_dataset.map(tokenize_function, batched=True)

    trainer = Trainer(
        model=model,
        train_dataset=tokenized_dataset,
        args=TrainingArguments(
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            warmup_steps=20,
            max_steps=200,          # 200 Steps = Deep Training
            learning_rate=2e-4,
            fp16=True,
            logging_steps=10,
            output_dir="outputs_enterprise",
            optim="paged_adamw_8bit"
        ),
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )

    print("🔥 STARTING TRAINING (This creates your new V2 Brain)...")
    trainer.train()
    print("✅ VICTORY: Quokka Enterprise V2 is Ready!")

🚀 Starting Enterprise V2 Training...
📂 Loading data from 'industry_data' folder...
✅ Data Loaded: 931 chunks ready for deep learning.
🧠 Loading Base Llama 3 Model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

...

🔥 STARTING TRAINING (This creates your new V2 Brain)...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,2.322856
20,2.193802
30,2.195299
40,2.022862
50,2.054850
60,2.052489
70,2.082233


Step,Training Loss
10,2.322856
20,2.193802
30,2.195299
40,2.022862
50,2.054850
60,2.052489
70,2.082233
80,2.177799
90,2.047789
100,2.099828


✅ VICTORY: Quokka Enterprise V2 is Ready!


In [ ]:
from google.colab import drive
import os

print("💾 Saving Enterprise V2 Model to Google Drive...")

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Define the PRO Save Path
save_path = "/content/drive/MyDrive/Quokka_Enterprise_v2"

# 3. Save the Adapter
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ SUCCESS: Enterprise Model saved at: {save_path}")
print("Ready to launch the App!")


💾 Saving Enterprise V2 Model to Google Drive...
Mounted at /content/drive
✅ SUCCESS: Enterprise Model saved at: /content/drive/MyDrive/Quokka_Enterprise_v2
Ready to launch the App!


In [2]:
# --- 1. MEMORY FLUSH & INSTALLS ---
import os
print("🛠️ Starting Agentic API Server (V13)...")
os.system("pip install -q -U git+https://github.com/huggingface/transformers.git")
os.system("pip install -q -U git+https://github.com/huggingface/accelerate.git")
os.system("pip install -q fastapi uvicorn pyngrok nest-asyncio python-multipart bitsandbytes PyPDF2 duckduckgo-search")

import torch
import gc
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from pyngrok import ngrok
import uvicorn
import nest_asyncio
import PyPDF2
from duckduckgo_search import DDGS
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TextIteratorStreamer
from threading import Thread
import datetime
from typing import List, Dict

gc.collect()
torch.cuda.empty_cache()

# --- 2. LOAD THE BRAIN ---
print("🧠 Loading Core Intelligence...")
model_id = "unsloth/llama-3-8b-Instruct-bnb-4bit"
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto", low_cpu_mem_usage=True)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# --- 3. THE API ENGINE ---
app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

global_document_context = ""

@app.post("/upload")
async def upload_file(file: UploadFile = File(...)):
    global global_document_context
    try:
        reader = PyPDF2.PdfReader(file.file)
        global_document_context = "".join([page.extract_text() + "\n" for page in reader.pages])[:15000]
        return {"status": "success", "message": "Loaded"}
    except Exception as e:
        return {"status": "error", "message": str(e)}

class ChatRequest(BaseModel):
    message: str
    history: List[Dict[str, str]]
    use_web: bool
    temperature: float

def search_web(query):
    try:
        results = list(DDGS().text(query, max_results=3))
        return "\n".join([f"- {res['title']}: {res['body']}" for res in results]) if results else "No results."
    except:
        return "Search failed."

@app.post("/chat")
async def chat_endpoint(req: ChatRequest):
    global global_document_context
    torch.cuda.empty_cache()

    try:
        today = datetime.datetime.now().strftime("%B %d, %Y")
        # FIX: The Auto-Thinking Agent Prompt
        system_prompt = f"""You are Quokka Pro, an elite Enterprise AI. Today is {today}.
        CRITICAL INSTRUCTION: Analyze the user's query and automatically adapt your expertise.
        - If the query is about C++ or Linux, act as a Senior Systems Engineer providing flawless, optimized code.
        - If the query is about Material Science or Perovskites, act as a PhD Researcher providing academic precision.
        - Otherwise, act as a concise, highly capable AI assistant.
        Always use Markdown, bold text, and code blocks."""

        if global_document_context.strip():
            system_prompt += f"\n\n--- DOCUMENT CONTEXT ---\n{global_document_context}\nAnswer strictly using this context."

        if req.use_web:
            system_prompt += f"\n\n--- WEB RESULTS ---\n{search_web(req.message)}\nSynthesize this data."

        messages = [{"role": "system", "content": system_prompt}] + req.history + [{"role": "user", "content": req.message}]
        prompt_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
        inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
        streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

        generation_kwargs = dict(**inputs, streamer=streamer, max_new_tokens=1024, temperature=req.temperature, do_sample=True)
        thread = Thread(target=model.generate, kwargs=generation_kwargs)
        thread.start()

        async def response_generator():
            for token in streamer:
                yield token
        # FIX: Event-stream prevents Ngrok from buffering the chunks
        return StreamingResponse(response_generator(), media_type="text/event-stream")

    except Exception as e:
        async def error_generator(): yield f"**[Error]** {str(e)}"
        return StreamingResponse(error_generator(), media_type="text/event-stream")

def run_server(): uvicorn.run(app, port=8000)

ngrok.kill()
ngrok.set_auth_token("39bfe2p03pjYle6le8VPArj8i5f_7wpmG7bsbjU1hpG58CbSa")
public_url = ngrok.connect(8000).public_url
print(f"\n🚀 API IS LIVE! Copy this URL:\n👉 {public_url}\n")
nest_asyncio.apply()
Thread(target=run_server).start()

🛠️ Starting Agentic API Server (V13)...
🧠 Loading Core Intelligence...


/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


🚀 API IS LIVE! Copy this URL:
👉 https://dissatisfiedly-cutcha-venice.ngrok-free.dev



INFO:     Started server process [291]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
